# Differential Equations — Session 14
## Section 4.2: Reduction of Order

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. Explain why a second independent solution is needed.
2. Derive the substitution $y_2=u y_1$.
3. Apply the reduction-of-order formula.
4. Track interval restrictions caused by zeros and singular coefficients.
5. Verify independence with the Wronskian.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–15 min | Motivation and quotient idea |\n| 15–38 min | Derivation of the method |\n| 38–60 min | Formula and worked examples |\n| 60–78 min | Interval restrictions and Wronskians |\n| 78–88 min | Symbolic/numerical verification |\n| 88–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

### Theorem 4.2-A — Reduction of order

Let $y_1$ be a nonzero solution on an interval $I$ of

$$
y''+P(x)y'+Q(x)y=0.
$$

A second solution is

$$
y_2=y_1\int \frac{e^{-\int P(x)\,dx}}{y_1(x)^2}\,dx.
$$

The formula is applied on an interval where $P,Q$ are continuous and $y_1\ne0$.

### Derivation idea

Set $y_2=u y_1$. After substitution and cancellation using the equation for $y_1$, the order drops because only $u'$ and $u''$ remain. Setting $w=u'$ creates a first-order equation.

### Independence check

If the integral factor is nonconstant, then $y_2/y_1=u$ is nonconstant. Equivalently, $W(y_1,y_2)\ne0$.

### Classroom Checkpoint — Reduction of Order

If $y_1(x)$ is a known nonzero solution of a second-order homogeneous linear equation, what trial form is used to seek a second solution?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Example: an Euler-type equation

Given $y_1=x$ for

$$
x^2y''-2xy'+2y=0,\qquad x>0,
$$

standard form has $P=-2/x$. The formula gives

$$
y_2=x\int\frac{e^{2\ln x}}{x^2}\,dx=x\int1\,dx=x^2.
$$

In [ ]:
x=sp.symbols('x',positive=True); y1=x; P=-2/x
y2=sp.simplify(y1*sp.integrate(sp.exp(-sp.integrate(P,x))/y1**2,x))
display(y2); display(wronskian_symbolic([y1,y2],x))

In [ ]:
xv=np.linspace(.1,3,500)
plt.plot(xv,xv,label='y1=x'); plt.plot(xv,xv**2,label='y2=x^2'); plt.plot(xv,(xv**2)/xv,linestyle='--',label='y2/y1=x')
plt.legend(); plt.title('A nonconstant quotient signals independence'); plt.show()

## 2. Example with interval restrictions

For $y''+y=0$, take $y_1=\cos x$. On any interval avoiding zeros of $\cos x$,

$$
y_2=\cos x\int\sec^2x\,dx=\sin x.
$$

The resulting $\sin x$ extends globally, although the derivation was local.

In [ ]:
x=sp.symbols('x',real=True); y1=sp.cos(x); P=0
y2=sp.simplify(y1*sp.integrate(1/y1**2,x)); display(y2); display(wronskian_symbolic([sp.cos(x),sp.sin(x)],x))

## 3. Interactive linear combinations

Once $y_1,y_2$ form a fundamental set, every solution is $c_1y_1+c_2y_2$.

In [ ]:
def combination(c1=1.0,c2=1.0):
    x=np.linspace(.15,4,600); y=c1*x+c2*x**2
    plt.plot(x,y,linewidth=2); plt.plot(x,c1*x,linestyle='--'); plt.plot(x,c2*x**2,linestyle=':'); plt.show()
if WIDGETS_AVAILABLE: interact(combination,c1=FloatSlider(min=-3,max=3,step=.25,value=1),c2=FloatSlider(min=-3,max=3,step=.25,value=1))
else: combination()

## 4. Numerical verification

A candidate second solution should satisfy both the differential equation and a nonzero Wronskian.

In [ ]:
x=sp.symbols('x',positive=True); y=sp.Function('y')
for f in [x,x**2]: display(sp.simplify(x**2*sp.diff(f,x,2)-2*x*sp.diff(f,x)+2*f))

## Classroom Checkpoint — Exit Check

Why must the formula be used on an interval where $y_1\ne0$?

> Pause here. Let students commit to an answer before running the next cell.